In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.bronze;

In [0]:
dbutils.widgets.text("table_name", "")
table_name = dbutils.widgets.get("table_name")

# Paths
source_path = f"abfss://data@datastoragea.dfs.core.windows.net/staging/{table_name}/"
schema_location = f"abfss://data@datastoragea.dfs.core.windows.net/bronze/{table_name}/schema/"
checkpoint_path = f"abfss://data@datastoragea.dfs.core.windows.net/bronze/{table_name}/checkpoint/"

print(f"table_name: '{table_name}'")
print(f"source_path:  '{source_path}'")
print(f"bronze_table: 'catalog.bronze.{table_name}'")

from pyspark.sql.functions import col, current_timestamp, input_file_name, to_date, expr

print("STARTING STREAM FOR:", table_name)
print("SOURCE:", source_path)
print("CHECKPOINT:", checkpoint_path)
print("SCHEMA:", schema_location)

# Autoloader 
df = (spark.readStream
            .format("cloudFiles")       
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")      # detects data types and saves to schemaLocation for reuse
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # new columns that appear in incoming files will be automatically added to the schemaLocation
            .option("cloudFiles.maxFilesPerTrigger", 1)         # Read 1 file at a time
            .option("cloudFiles.schemaLocation", schema_location)
            # .option("cloudFiles.includeExistingFiles", "true")  # load past data on first run
            .load(source_path)
    )

# Add metadata columns
df_with_metadata = (df
    .withColumn("_source_file", col("_metadata.file_path"))    
    .withColumn("_ingest_timestamp",current_timestamp())
    .withColumn("ingestion_date",   to_date("_ingest_timestamp"))
)

# Write Bronze layer - Append
(df_with_metadata
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)     # ingests all available files and stops
    .option("mergeSchema", "true")       # allows addNewColumns to persist
    .toTable(f"catalog.bronze.{table_name}")
)


In [0]:
tables = ["diagnosis_raw", "visits_raw", "hospitals_raw", "patients_raw", "physicians_raw"]

for table_name in tables:
    # Load the table as a DataFrame and count rows
    full_table = f"catalog.bronze.{table_name}"
    row_count = spark.table(full_table).count()
    print(f"Table '{full_table}' has {row_count} rows.")

In [ ]:
print("Testing pipeline")